# AVL Tree (균형 이진 탐색 트리의 첫등장)
- 모든 노드에 대해서 노드의 왼쪽 부트리와 오른쪽 부트리의 높이차가 1이하인 BST
- 트리의 높이가 O(log N)에 비례하는 균형BST
- Rotation

In [1]:
from abc import ABC, abstractmethod

In [2]:
class AVLNode:
    def __init__(self, value):
        self.left:AVLNode = None
        self.right:AVLNode = None
        self.parent:AVLNode = None
        self.height:int = 0
        self.value = value
    
    @property
    def left_height(self):
        return -1 if self.left is None else self.left.height
    
    @property
    def right_height(self):
        return -1 if self.right is None else self.right.height
    
    @property
    def balance_factor(self):
        return self.left_height - self.right_height

In [3]:
class BaseAVLTree(ABC):
    @abstractmethod
    def insert(self, value):
        pass
    
    @abstractmethod
    def delete(self, value):
        pass
    
    @abstractmethod
    def search(self, value):
        pass
    
    @abstractmethod
    def rotate_right(self):
        pass
    
    @abstractmethod
    def rotate_left(self):
        pass

In [ ]:
class AVLTree:
    def __init__(self):
        self.root:AVLNode = None
        self._size = 0
        
    @property
    def size(self):
        return self._size
    
    def inorder(self, node:AVLNode):
        if node is None:
            return 
        
        yield from self.inorder(node.left)
        yield node
        yield from self.inorder(node.right)
    
    def __iter__(self):
        yield from self.inorder(self.root)    
    
    def insert(self, value):
        if self._size == 0:
            self.root = AVLNode(value)
            self._size += 1
            return self.root
        
        p = None
        v = self.root
        new_node = AVLNode(value)

        # 최적의 노드 장소 찾기    
        while v: # 현재 노드가 존재한다면,
            if v.value > value:
                p = v 
                v = v.left

            elif v.value < value:
                p = v 
                v = v.right

            else:
                print(f"중복은 허용되지 않습니다. {v} 반환")
                return v
                   
        # 삽입할 노드 적절한 곳에 삽입
        if p.value <= value:
            p.right = new_node
            new_node.parent = p
            
        elif p.value > value:
            p.left = new_node
            new_node.parent = p

        self._size += 1
        
        # 삽입된 노드의 부모 노드들 Height 업데이팅
        # 그리고 삽입 이후에 BF (균형 인자)가 무너졌을 수도 있음 - BF 체킹
        new_node_parent = new_node.parent
        while new_node_parent: # 루트 노드까지 진행
            self._update_height(new_node_parent) # 삽입된 노드의 부모노드 Height 업데이팅
            bf = self._balance_factor(new_node_parent)
            
            if bf > 1: # 왼쪽 부트리가 더 높이가 클 때, LL - LR
                z = new_node_parent
                y = z.left
                if self._balance_factor(y) >= 0: # LL
                    self._rotate_right(z)
                else:
                    self._rotate_left(y)
                    self._rotate_right(z)
                break
            
            elif bf < -1: # 오른쪽 부트리가 더 높이가 클 때, RR - RL
                z = new_node_parent
                y = z.right
                if self._balance_factor(y) <= 0: # RR
                    self._rotate_left(z)
                else:
                    self._rotate_right(y)
                    self._rotate_left(z)
                break
            new_node_parent = new_node_parent.parent
            
        return new_node

    def search(self, value):
        v = self.root # None or Node
        
        if not v: # 없으면
            print("루트 노드 존재하지 않습니다.")
            return None
        
        while v:
            if v.value == value:
                print(f"{value} 노드 검색 완료")
                return v
            elif v.value < value:
                v = v.right
            else:
                v = v.left
        print(f"{value} 노드는 존재하지 않습니다.")
        return None
        
    def _update_height(self, node:AVLNode):
        left_height = -1 if node.left is None else node.left.height
        right_height = -1 if node.right is None else node.right.height 
        node.height = max(left_height, right_height) + 1
        
    def _balance_factor(self, node: AVLNode):
        left_height = -1 if node.left is None else node.left.height
        right_height = -1 if node.right is None else node.right.height
        return left_height - right_height
            
    def delete(self, value:int):
        node = self._delete_by_merging(value)

        if node is self.root:
            return None

        while node:
            self._update_height(node)

            bf = self._balance_factor(node)

            if bf > 1:
                y = node.left

                if self._balance_factor(y) >= 0:
                    self._rotate_right(node)      # LL
                else:
                    self._rotate_left(y)          # LR
                    self._rotate_right(node)

            elif bf < -1:
                y = node.right

                if self._balance_factor(y) <= 0:
                    self._rotate_left(node)       # RR
                else:
                    self._rotate_right(y)         # RL
                    self._rotate_left(node)
            node = node.parent

    def _delete_by_merging(self, value):
        p, v = None, self.root
        
        while v:
            if v.value < value:
                p = v
                v = v.right
            elif v.value > value:
                p = v
                v = v.left
            else: # 삭제할 노드 찾음
                break
        else:
            print(f"삭제할 {value} 노드를 찾지 못했습니다.")
            return None
    
        if v.left is None and v.right is None: # 삭제 노드가 리프 노드 일 때
            if p is None:
                self._size -= 1
                self.root = None
                return self.root
            
            elif p.left is v:
                p.left = None
                self._size -= 1
                return p
            
            elif p.right is v:
                p.right = None
                self._size -= 1
                return p
        
        # 삭제 노드의 자식 노드가 1개 있을 때
        elif v.left and v.right is None: # 왼쪽 자식 노드만
            if p is None: # 삭제 노드가 루트노드일 때,
                self._size -= 1
                self.root = v.left
                v.left.parent = None
                return p
            
            elif p.left is v: # Ok
                self._size -= 1 
                p.left = v.left
                v.left.parent = p
                return p
                
            elif p.right is v: # OK
                self._size -= 1 
                p.right = v.left
                v.left.parent = p
                return p
            
        elif v.right and v.left is None: # 오른쪽 자식 노드만
            if p is None: # 삭제 노드가 루트노드라면,
                self._size -= 1
                self.root = v.right
                v.right.parent = None
                return self.root

            elif p.left is v:
                self._size -= 1
                p.left = v.right
                v.right.parent = p
                return p
                        
            elif p.right is v:
                self._size -= 1
                p.right = v.right
                v.right.parent = p
                return p
            
        # 삭제 노드의 자식 노드가 2개 있을 때
        elif v.right and v.left:
            # 1. m 찾기 (왼쪽 서브트리에서 가장 큰 값)
            m = v.left
            while m.right:
                m = m.right
            
            # 2. m의 오른쪽에 v의 오른쪽 서브트리를 붙이기
            m.right = v.right
            v.right.parent = m

            # 3. 이제 v의 부모(p)와 v의 왼쪽 자식(v.left를 직접 연결)
            if p is None: # 삭제 노드가 루트노드인 경우
                self.root = v.left
                v.left.parent = None
                return self.root
            
            elif p.left is v:
                p.left = v.left
                v.left.parent = p
                
            elif p.right is v:
                p.right = v.left
                v.left.parent = p 

            self._size -= 1
            return p

    def _rotate_left(self, z:AVLNode):
        y = z.right
        z.right = y.left
        
        if not z.parent:
            self.root = y
            y.parent = None
            
        elif z.parent.left == z:
            z.parent.left = y
            y.parent = z.parent
        
        elif z.parent.right == z:
            z.parent.right = y
            y.parent = z.parent
        
        y.left = z
        z.parent = y

    def _rotate_right(self, z:AVLNode):
        y = z.left
        z.left = y.right
        
        if not z.parent:
            self.root = y
            y.parent = None
        
        elif z.parent.right == z:
            z.parent.right = y
            y.parent = z.parent
            
        elif z.parent.left == z:
            z.parent.left = y
            y.parent = z.parent
            
        y.right = z
        z.parent = y

In [5]:
avl = AVLTree()

avl.insert(20)
avl.insert(15)
avl.insert(10)
avl.insert(13)

avl.delete(15)

for node in avl:
    print(f"Node: {node.value} - height: {node.height}")

Node: 10 - height: 1
Node: 13 - height: 0
Node: 20 - height: 2
